# 1 - Forward modeling + autodiff gradient (aluminum plate)

Forward-model the 200x100 mm aluminum plate (200 kHz Gaussian-derivative source) and get the FWI gradient `dJ/d(alpha2)` by autodiff. Thesis-faithful port of `InversionToolbox/ndt`.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/seidlr/acoustic-fwi-ndt-pytorch/blob/main/notebooks/01_autodiff_fwi.ipynb)

In [ ]:
import sys, os

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    OWNER = "seidlr"  # change to your account if you forked
    !git clone -q https://github.com/{OWNER}/acoustic-fwi-ndt-pytorch.git
    %cd acoustic-fwi-ndt-pytorch
    sys.path.insert(0, os.path.abspath("src"))  # make `import fwi` importable now

import torch
from IPython.display import Image, display
from fwi.config import resolve_device, resolve_dtype

device = resolve_device()
dtype = resolve_dtype(device)
print("device:", device, "| dtype:", dtype)

## Forward: homogeneous vs cracked plate

In [ ]:
from fwi.problems import build_problem
from fwi.forward import forward
from fwi.misfit import l2_misfit
from fwi import plotting

prob = build_problem('crack', device=device, dtype=dtype)
res = forward(prob.true_alpha2, prob.src_sig, prob.src_i, prob.src_j,
              prob.rec_i, prob.rec_j, prob.cfg, capture_wavefield=True)
display(Image(str(plotting.save_field(res.wavefield[prob.cfg.nt//3],
    'outputs/nb1_wavefield.png', title='aluminum wavefield'))))
syn = forward(prob.start_alpha2, prob.src_sig, prob.src_i, prob.src_j,
              prob.rec_i, prob.rec_j, prob.cfg).traces
print('start-model misfit (sane, not the old 1e-50):',
      float(l2_misfit(syn, prob.observed, prob.cfg.dt)))

## Autodiff gradient dJ/d(alpha2)

In [ ]:
from fwi.misfit import autodiff_gradient
grad, J = autodiff_gradient(prob.start_alpha2, prob.observed, prob.src_sig,
    prob.src_i, prob.src_j, prob.rec_i, prob.rec_j, prob.cfg, active_mask=prob.active_mask)
print(f'misfit J = {J:.3e}')
display(Image(str(plotting.save_field(grad, 'outputs/nb1_kernel.png',
    title='autodiff dJ/d(alpha2)'))))